In [ ]:


import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt



# 2. Load Dataset

cancer = load_breast_cancer()

X = cancer.data
y = cancer.target

print("Dataset Shape:", X.shape)
print("Target Names:", cancer.target_names)


# 3. Split Dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)


# 4. Feature Scaling

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)



class KNN:

    # Constructor
    def __init__(self, k=5):
        self.k = k

    # Calculate Euclidean Distance
    def euclidean_distance(self, point1, point2):

        distance = np.sqrt(np.sum((point1 - point2) ** 2))

        return distance

    # Train the Model
    def fit(self, X_train, y_train):

        self.X_train = X_train
        self.y_train = y_train

    # Predict Class of One Data Point
    def predict_one(self, test_point):

        distances = []

        # Calculate distance from every training point

        for i in range(len(self.X_train)):

            distance = self.euclidean_distance(
                test_point,
                self.X_train[i]
            )

            distances.append((distance, self.y_train[i]))

        # Sort distances in ascending order

        distances.sort(key=lambda x: x[0])

        # Select K nearest neighbors

        k_nearest = distances[:self.k]

        # Get the classes of K neighbors

        labels = []

        for distance, label in k_nearest:

            labels.append(label)

        # Majority Voting

        prediction = max(set(labels), key=labels.count)

        return prediction

    # Predict for Multiple Data Points
    def predict(self, X_test):

        predictions = []

        for test_point in X_test:

            result = self.predict_one(test_point)

            predictions.append(result)

        return np.array(predictions)


# ============================================
# 6. CREATE KNN MODEL
# ============================================

model = KNN(k=5)


# 7. Train Model

model.fit(X_train, y_train)

print("\nModel Training Completed!")


# 8. Make Predictions

y_pred = model.predict(X_test)


# ============================================
# 9. EVALUATE MODEL
# ============================================

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", accuracy * 100, "%")


# 10. Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=cancer.target_names,
    yticklabels=cancer.target_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("KNN Confusion Matrix")

plt.show()


# 11. Classification Report

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=cancer.target_names
    )
)


# ============================================
# 12. FIND BEST K VALUE
# ============================================

k_values = range(1, 21)

accuracy_values = []

for k in k_values:

    model = KNN(k=k)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    accuracy_values.append(accuracy)


# 13. Plot K vs Accuracy

plt.figure(figsize=(8, 5))

plt.plot(
    k_values,
    accuracy_values,
    marker='o'
)

plt.xlabel("K Value")
plt.ylabel("Accuracy")

plt.title("KNN: K Value vs Accuracy")

plt.xticks(k_values)

plt.grid()

plt.show()


# 14. Find Best K

best_k = list(k_values)[np.argmax(accuracy_values)]

print("\nBest K Value:", best_k)

print(
    "Best Accuracy:",
    max(accuracy_values) * 100,
    "%"
)
